# Filtrando Grupos: HAVING e ANY

---

## Contexto: perguntas sobre totais e comparações

O gerente comercial chegou com novas perguntas e dessa vez elas envolvem **grupos** e **comparações com conjuntos**:

- *"Quais categorias têm mais de 3 produtos cadastrados?"*
- *"Quais clientes fizeram mais de 2 pedidos no total?"*
- *"Quais produtos têm preço maior que pelo menos um item vendido abaixo de R$50?"*

Para as duas primeiras, usamos **`HAVING`** que filtra *grupos* depois do `GROUP BY`.  
Para a terceira, usamos **`ANY`** que compara um valor com *qualquer elemento* de um conjunto.

Nesta aula você vai aprender:

- `HAVING` — filtrar resultados agrupados com condições sobre agregações
- `ANY` — comparar um valor com ao menos um elemento de uma subconsulta

---

## 1. Configuração

In [ ]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

In [ ]:
# Verificar tabelas com SQLAlchemy
insp = inspect(engine)

print(insp.get_table_names())

# 2. HAVING

O `HAVING` filtra grupos depois do `GROUP BY`, da mesma forma que o `WHERE` filtra linhas individuais, mas aplicado sobre o resultado de uma agregação.
**Por exemplo:** se você agrupar pedidos por cliente e contar quantos cada um fez, o `HAVING` permite manter só os grupos onde essa contagem é maior que 2. Isso não é possível com WHERE, porque a contagem ainda não existe quando o WHERE é aplicado.

Resumindo: `WHERE` filtra antes de agrupar, `HAVING` filtra depois.

In [ ]:
# código

---

## 2. HAVING — filtrando grupos

### WHERE vs HAVING: qual é a diferença?

Essa é a confusão mais comum em SQL. A regra é simples:

| | `WHERE` | `HAVING` |
|---|---|---|
| Filtra | Linhas individuais | Grupos (resultado do GROUP BY) |
| Aplicado | **Antes** da agregação | **Depois** da agregação |
| Pode usar `func.count()`, `func.sum()`? | ❌ Não | ✅ Sim |

**Analogia:** imagine que você está contando votos por candidato.  
- `WHERE` descarta cédulas inválidas *antes* de contar.  
- `HAVING` elimina candidatos com poucos votos *depois* de contar.

### Caso de uso: categorias com mais de 3 produtos

## 2.1 Combinando WHERE e HAVING na mesma query

Os dois podem coexistir. `WHERE` filtra as linhas brutas; `HAVING` filtra os grupos resultantes.

**Caso de uso:** clientes ativos (com pedido não cancelado) que fizeram mais de 2 pedidos.

In [ ]:
# código:

## 2.2 Recuperando os objetos completos após o HAVING

O `HAVING` retorna apenas os campos do `GROUP BY`. Para buscar os objetos `Cliente` completos, basta usar `.in_()` com os IDs filtrados é um padrão que já vimos na aula anterior.

In [ ]:
# código:


**O padrão é sempre este:**
```
1. Montar a query com GROUP BY + HAVING   →  select(...).group_by(...).having(...)
2. Usar como subconsulta                  →  .in_(subconsulta)
3. Buscar os objetos completos            →  select(Modelo).where(...)
```

---

## 3. ANY — comparar com ao menos um valor do conjunto

### O que é ANY?

`ANY` responde: *"este valor é maior/menor/igual a **pelo menos um** valor dessa subconsulta?"*

Funciona com qualquer operador de comparação: `>`, `<`, `=`, `!=`, `>=`, `<=`.

| Expressão | Significado |
|---|---|
| `col > any_(subq)` | `col` é maior que **ao menos um** valor da subconsulta |
| `col = any_(subq)` | `col` é igual a **ao menos um** valor (equivale ao `IN`) |
| `col < any_(subq)` | `col` é menor que **ao menos um** valor |


> ⚠️ **Nota SQLite:** `ANY` é um operador do padrão SQL e funciona nativamente em PostgreSQL e MySQL. No SQLite, o SQLAlchemy emula o comportamento por baixo para produção com SQLite, prefira `IN` ou `EXISTS`.

### Caso de uso: produtos com preço acima de algum item vendido a preço baixo

In [ ]:
# código:

In [ ]:
# códigos:

---

## Exercício: Usando IA para isso

**Prompt para gerar um HAVING:**
```
Com os modelos SQLAlchemy abaixo, escreva uma query que retorne
os estados com mais de 2 clientes cadastrados, ordenados
pelo total de clientes decrescente.
Use GROUP BY + HAVING.

[modelos ORM]
```
---

### Resposta:Código gerado pelo Copilot

In [ ]:
# Código para retornar estados com mais de 2 clientes cadastrados, ordenados pelo total decrescente
stmt = (
    select(
        Cliente.estado,  # Seleciona o estado
        func.count(Cliente.id).label("total_clientes")  # Conta o número de clientes por estado
    )
    .group_by(Cliente.estado)  # Agrupa por estado
    .having(func.count(Cliente.id) > 2)  # Filtra grupos com mais de 2 clientes
    .order_by(func.count(Cliente.id).desc())  # Ordena pelo total de clientes decrescente
)

# Executa a consulta
with Session(engine) as session:
    resultados = session.execute(stmt).all()
    for estado, total in resultados:
        print(f"{estado}: {total} clientes")